# Basic Distillation

This notebook demonstrates the full softlabels pipeline:
- **Broker** routes requests between clients and workers
- **Worker** generates batches of soft labels
- **Client** trains a student model on those labels

We use random data here to keep it fast. For real LLM distillation, see `examples/distill_llm.py`.

In [1]:
import time
import threading
import torch
import torch.nn as nn

from softlabels import (
    Broker, Worker, Client,
    BatchConfig, TensorSpec, EndpointConfig,
)

## 1. Define the batch layout

Each slot holds one batch: `x` (inputs) and `y` (soft targets).

In [2]:
BATCH_SIZE = 8
HIDDEN = 256
NUM_CLASSES = 100

config = BatchConfig([
    TensorSpec("x", (BATCH_SIZE, HIDDEN), "float32"),
    TensorSpec("y", (BATCH_SIZE, NUM_CLASSES), "float32"),
])

print(f"Slot size: {config.nbytes() / 1e6:.1f} MB")

Slot size: 0.1 MB


## 2. Start the broker

In [3]:
endpoints = EndpointConfig()
broker = Broker(endpoints=endpoints)
broker.start()
time.sleep(0.2)

## 3. Start a worker

The worker generates random soft labels. In practice, this would run a teacher model.

In [ ]:
def teacher_generator(model_id: str) -> bytes:
    """Simulate teacher: random soft targets."""
    x = torch.randn(BATCH_SIZE, HIDDEN)
    logits = torch.randn(BATCH_SIZE, NUM_CLASSES)
    y = torch.softmax(logits, dim=-1)  # soft targets
    return config.encode(x=x, y=y)

worker = Worker(
    generator_fn=teacher_generator,
    model_ids=["teacher_v1"],
    slot_size=config.nbytes(),
    endpoints=endpoints,
)
worker.start()
time.sleep(0.2)

## 4. Create a client and train

The client requests batches by `model_id`, reads them from shared memory, and trains a small student.

In [5]:
# Simple student model
student = nn.Sequential(
    nn.Linear(HIDDEN, 128),
    nn.ReLU(),
    nn.Linear(128, NUM_CLASSES),
)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

client = Client(slot_count=4, batch_config=config, endpoints=endpoints)
client.hello()

for step in range(10):
    slot = client.request_sample("teacher_v1", timeout_ms=5000)
    assert slot is not None, f"Timeout at step {step}"
    
    batch = config.decode(client.medium.read(slot))
    client.release_slot(slot)
    
    x, y_teacher = batch["x"], batch["y"]
    y_student = torch.log_softmax(student(x), dim=-1)
    loss = -(y_teacher * y_student).sum(dim=-1).mean()
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Step {step:2d}: loss={loss.item():.4f}")

Step  0: loss=4.6154
Step  1: loss=4.6060
Step  2: loss=4.6067
Step  3: loss=4.5955
Step  4: loss=4.5993
Step  5: loss=4.5893
Step  6: loss=4.5949
Step  7: loss=4.5770
Step  8: loss=4.5887
Step  9: loss=4.5779


## 5. Check broker stats

In [6]:
stats = broker.get_stats()
print(f"Workers: {stats.connected_workers}, Clients: {stats.connected_clients}, Completed: {stats.total_completed}")

Workers: 1, Clients: 1, Completed: 10


## 6. Graceful shutdown

Order matters: close client first, then stop worker (sends GOODBYE), then stop broker.

In [7]:
client.close()
worker.stop()
time.sleep(0.2)
broker.stop()
print("Shutdown complete.")

Shutdown complete.
